# Event-by-Event Mean $p_T$ Distribution — Gamma Fit

Gamma-distribution fit to the event-by-event mean-$p_T$ spectra from
Trajectum HDF5 output files.

**Reads:** `eventbyeventmeanptcharged/<track-cut group>/centralitybinned`
(track-cut group is auto-detected per file, e.g. `STARTPC` or `STARTPC200MeV`)

**Fits:** `A * gamma.pdf(x, a, loc, scale)` to each centrality bin, in each file

**Plots:** the fitted curves only (log-y), zoomed to where the data lives.
Color encodes file, linestyle encodes centrality.

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.optimize import curve_fit
from scipy.stats import gamma


In [ ]:
# ------------------------------------------------------------------
# 1. Files to scan.
#
# Each file is a Trajectum pT-study output. The track-cut group name
# (e.g. "STARTPC" vs "STARTPC200MeV") differs between file types, so
# it is auto-detected per file below rather than hardcoded.
# ------------------------------------------------------------------
filepaths = [
    "src/monotonic_ptfluc_study/7.7GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/19GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/27GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/54GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/200GeV_200MeVcut_ptfluc_study.h5",


]

XLIM = (0.45, 0.85)
YLIM = (1e-4, 40)
OUTPUT_PNG = "graphs/eventbyevent_meanpt_dist.png"


In [ ]:
def _detect_track_group(hdf, base="eventbyeventmeanptcharged"):
    """Return the single track-cut group name under `base`
    (e.g. "STARTPC" or "STARTPC200MeV"). Raises if the file contains
    more than one, in which case pass track_group explicitly to
    get_gamma_fits_from_file."""
    groups = list(hdf[base].keys())
    if len(groups) != 1:
        raise ValueError(
            f"Expected exactly one track-cut group under '{base}', "
            f"found {groups}. Pass track_group explicitly."
        )
    return groups[0]


def _file_label(filepath):
    return os.path.splitext(os.path.basename(filepath))[0]


In [ ]:
def gamma_func(x, A, a, loc, scale):
    """Scaled gamma PDF used as the fit model."""
    return A * gamma.pdf(x, a, loc=loc, scale=scale)


def fit_gamma_to_bin(binc, y):
    """Fit a scaled gamma PDF to one centrality bin's distribution."""
    mask = y > 1e-6  # ignore near-zero/noise bins for the fit
    mean_guess = binc[np.argmax(y)]

    p0 = [1.0, 100, 0.0, mean_guess / 100]
    bounds = (
        [0, 1, -0.5, 1e-5],               # lower bounds: A, a, loc, scale
        [1000, 10000, binc.min(), 1.0],   # upper bounds
    )

    popt, pcov = curve_fit(
        gamma_func, binc[mask], y[mask], p0=p0, bounds=bounds, maxfev=50000
    )
    return popt  # A, a, loc, scale


In [ ]:
def get_gamma_fits_from_file(filepath, track_group=None):
    """
    Returns records:
      (file_label, cent_label, binc, popt)

    track_group: name of the track-cut group to read (e.g. "STARTPC",
    "STARTPC200MeV"). If None (default), it is auto-detected from the
    file. The file's basename (without extension) is used as
    file_label, so points from different files can be told apart
    downstream (e.g. colored separately in a plot).
    """
    label = _file_label(filepath)
    records = []

    with h5py.File(filepath, "r") as f:
        grp = track_group or _detect_track_group(f)
        data_group = f"eventbyeventmeanptcharged/{grp}/centralitybinned"

        g = f[data_group]
        binc = g["bin"][:]            # <pT> bin centers [GeV]
        vals = g["values"][:]         # shape (n_centrality_bins, 1, n_bins)
        cent = f["centrality"][:]     # bin centers [%]
        dcent = f["dcentrality"][:]   # bin half-widths [%]

        for i in range(vals.shape[0]):
            y = vals[i, 0, :]
            c_lo = cent[i] - dcent[i]
            c_hi = cent[i] + dcent[i]
            cent_label = f"{c_lo:.0f}-{c_hi:.0f}%"

            popt = fit_gamma_to_bin(binc, y)
            records.append((label, cent_label, binc, popt))

    return records


In [ ]:
# ------------------------------------------------------------------
# 2. Collect fits from all files.
# ------------------------------------------------------------------
all_fits = []
for fp in filepaths:
    all_fits.extend(get_gamma_fits_from_file(fp))

for file_label, cent_label, binc, popt in all_fits:
    A, a, loc, scale = popt
    mean = loc + a * scale
    sigma = np.sqrt(a) * scale
    skew = 2 / np.sqrt(a)
    print(
        f"{file_label} [{cent_label}]: A={A:.4f}  a={a:.2f}  loc={loc:.5f}  "
        f"scale={scale:.6f}  -> mean={mean:.4f} GeV, sigma={sigma:.4f} GeV, "
        f"skew={skew:.4f}"
    )


In [ ]:
# ------------------------------------------------------------------
# 3. Plot: fitted curves only, log-y.
#
# Color encodes file; linestyle encodes centrality. A single combined
# legend shows both dimensions together (correct color + correct
# linestyle per entry).
#
# Tick/grid styling below (inward ticks on all sides, light grid,
# serif/CM math text) matches the STAR-style look used elsewhere.
# ------------------------------------------------------------------
plt.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.size": 12,
})

fig, ax = plt.subplots(figsize=(8, 6), dpi=200)

file_labels_seen = sorted({fl for (fl, _, _, _) in all_fits})
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_map = {fl: color_cycle[i % len(color_cycle)] for i, fl in enumerate(file_labels_seen)}

cent_labels_seen = sorted({cl for (_, cl, _, _) in all_fits})
linestyle_cycle = ["-", "--", ":", "-."]
linestyle_map = {cl: linestyle_cycle[i % len(linestyle_cycle)] for i, cl in enumerate(cent_labels_seen)}

sorted_fits = sorted(all_fits, key=lambda r: (r[0], r[1]))
legend_handles = []

for file_label, cent_label, binc, popt in sorted_fits:
    color = color_map[file_label]
    linestyle = linestyle_map[cent_label]

    xfit = np.linspace(binc.min(), binc.max(), 1000)
    ax.plot(xfit, gamma_func(xfit, *popt), color=color, linestyle=linestyle, lw=2)

    legend_handles.append(
        plt.Line2D([0], [0], color=color, linestyle=linestyle, lw=2,
                   label=f"{file_label} - {cent_label}")
    )

ax.set_xlabel(r"$\langle p_T \rangle$  [GeV]")
ax.set_ylabel(r"P($\langle p_T \rangle$)  [GeV$^{-1}$]")
ax.set_title(
    "Event-by-Event Mean $p_T$ Distribution (charged, 0.2-2 GeV, "
    "|$\\eta$|<0.5)\nAu+Au 200 GeV - Gamma Fit"
)

ax.set_xlim(*XLIM)
ax.set_yscale("log")
ax.set_ylim(*YLIM)

ax.xaxis.set_major_locator(ticker.MultipleLocator(0.05))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.01))

# Inward ticks on all four sides (STAR-ish)
ax.tick_params(axis="both", which="major", length=7, width=1.1, direction="in", top=True, right=True)
ax.tick_params(axis="both", which="minor", length=4, width=0.9, direction="in", top=True, right=True)

# Light grid, major + minor
ax.grid(which="major", alpha=0.3)
ax.grid(which="minor", alpha=0.15)

# Single combined legend: color -> file, linestyle -> centrality
ax.legend(handles=legend_handles, loc="best", fancybox=True)

fig.tight_layout()
fig.savefig(OUTPUT_PNG, dpi=150)
print(f"saved -> {OUTPUT_PNG}")
plt.show()
